# Signal-to-noise maps

We estimate the noise of a mass map from randomized copies of its catalog, check the estimate on a catalog
without lensing signal, compare the two ways SMPy randomizes a catalog, use the B mode as a null test and find the
peaks of a signal-to-noise map.

## Settings

Paths are relative to `docs/source/tutorials` in the SMPy repository.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from scipy.stats import norm

import smpy
from smpy.utils import find_peaks2d

CATALOG = Path("../../../examples/data/forecast_lum_annular.fits")
SETTINGS = dict(pixel_scale=0.4, g1_col="g1_Rinv", g2_col="g2_Rinv", weight_col="weight")

## Noise from randomized catalogs

Turning every galaxy's shear by an independent random angle removes the coherent lensing signal and keeps each
galaxy's position, weight and shear amplitude. Maps of such catalogs sample the noise of the map, and SMPy divides
the map by their standard deviation in each pixel,

$$
\mathrm{SNR}(\boldsymbol\theta) = \frac{\kappa(\boldsymbol\theta)}{\sigma_\mathrm{null}(\boldsymbol\theta)}.
$$

The randomized catalogs, `num_shuffles` of them (100 by default), are mapped with the same method and settings as
the catalog.

In [ ]:
result = smpy.map_mass(CATALOG, create_snr=True, **SETTINGS)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"wspace": 0.4})
result.plot("E", snr=True, ax=axes[0], cmap="magma")
result.plot("B", snr=True, ax=axes[1], cmap="magma")
plt.show()

The SNR is a detection statistic: it measures how unusual a pixel is next to maps without signal. Because the
randomized catalogs keep each galaxy's shear amplitude, which includes the lensing signal, the noise near a strong
lens is slightly overestimated. For KS+, whose reconstruction depends on the signal, the spread of maps without
signal need not equal the uncertainty of a map with signal.

## Calibration

A catalog whose own shears are randomized has no signal, so its SNR map should have zero mean and unit spread.
Neighboring pixels are correlated, so the spread of a single map scatters by several percent around one.

In [ ]:
catalog = Table.read(CATALOG, hdu=1)
turn = np.exp(2j * np.random.default_rng(1).uniform(0, np.pi, len(catalog)))
shear = (catalog["g1_Rinv"] + 1j * catalog["g2_Rinv"]) * turn
null = catalog.copy()
null["g1_Rinv"], null["g2_Rinv"] = shear.real, shear.imag

snr = smpy.map_mass(null, create_snr=True, **SETTINGS).snr["E"]
print(f"Mean {snr.mean():+.2f}, spread {snr.std():.2f}")

bins = np.linspace(-5, 5, 41)
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(snr.ravel(), bins, density=True, histtype="step", label="Catalog without signal")
ax.plot(bins, norm.pdf(bins), "k--", label="Unit normal")
ax.set_xlabel("E-mode SNR")
ax.legend()
plt.show()

## Orientation and spatial shuffles

With `shuffle_type: 'spatial'`, SMPy instead permutes the galaxy positions, moving each shear and weight to another
galaxy's position. This keeps the noise of each pixel only if the noise is the same across the field. Here we
triple the shear of the eastern half of the catalog without signal: orientation shuffles follow the change,
while spatial shuffles average the noise over the field and misjudge both halves.

In [ ]:
noisy = null.copy()
east = noisy["ra"] > np.median(noisy["ra"])
for column in ("g1_Rinv", "g2_Rinv"):
    noisy[column][east] *= 3

for shuffle_type in ("orientation", "spatial"):
    snr = smpy.map_mass(noisy, create_snr=True, snr={"shuffle_type": shuffle_type}, **SETTINGS).snr["E"]
    east_half, west_half = np.array_split(snr, 2, axis=1)
    print(f"{shuffle_type}: SNR spread {east_half.std():.2f} in the east, {west_half.std():.2f} in the west")

## Number of shuffles and seed

The noise estimated from $N$ randomized catalogs has a relative uncertainty of about $1/\sqrt{2(N - 1)}$, 7% for
100. The randomization is seeded, by default with 0, so runs repeat exactly; `seed: 'random'` draws a seed and
records it in the settings of the result.

In [ ]:
seeded = smpy.map_mass(CATALOG, create_snr=True, snr={"seed": "random", "num_shuffles": 20}, **SETTINGS)
seeded.settings["snr"]["seed"]

## The B mode as a null test

Lensing produces no B modes, so the B-mode SNR of a catalog should be distributed like the SNR of a catalog
without signal. An excess points to systematic errors in the shear.

In [ ]:
print(f"B-mode SNR: mean {result.snr['B'].mean():+.2f}, spread {result.snr['B'].std():.2f}")

## Peaks

Local maxima of the E-mode SNR map above a threshold are candidate mass concentrations. `find_peaks2d` returns their
pixel positions and heights, and the WCS of the map gives their positions on the sky. The `threshold` plotting
setting circles them on the map.

In [ ]:
x, y, heights, _ = find_peaks2d(result.snr["E"], threshold=3)
for position, height in zip(result.wcs.pixel_to_world(x, y), heights):
    print(f"{position.to_string('hmsdms', precision=1)}  SNR {height:.1f}")

result.plot("E", snr=True, cmap="magma", threshold=3)
plt.show()